# Task A：双寄存器条件更新校准 v3

目标：先跑 `d=1 × 100 × DA` 探针，再跑 `d∈{1,5,10,15,20} × 20 × DA/CoT`，
人工审查后才生成正式数据。**已提交的 Colab 运行（87911b7）仅完成离线自测，尚无真实模型结果。**
本版默认 `RUN_STAGE="probe"`，会加载 T4 上的 4B 模型并运行 100 条真实 DA 探针。
只有明确选择 `offline` 才跳过推理；离线自测成功不代表任务已通过校准。
只用 `answer_ok` 评估问题集，不筛约束遵循，不运行 Task B 或最终 k 矩阵。

执行顺序：
1. Colab 选择 **T4 GPU**，保留 `RUN_STAGE="probe"`，从头运行；依赖准备 cell 会补装缺失包。
2. 跑首选候选的 100 题，查看 p1 与 Wilson 区间；末尾导出真实原文和摘要 ZIP。
3. 若过易/过难，改 `PROBE_CANDIDATES` 逐个测试预设或新配置；不能按预估值冒充实测。
4. 指定 `SELECTED`，仅对已完成探针的配置设 `RUN_STAGE="slope"`，跑独立小样本并查看摘要和原文。
5. 人工审查后填写 `REVIEW`，设 `GENERATE_FINAL=True` 才写正式 JSONL。

每个阶段可以单独重跑，已有结果按题目、提示词、模型、采样参数和随机种子的哈希去重。
修改生成/统计语义时须增加 `VERSION`；恢复运行须保持相同软件与模型版本。

## Colab / T4 准备

在 Colab 选择 **T4 GPU**，可直接从 GitHub 打开本文件；无需先克隆仓库。
配置 cell 优先使用 `TASK_A_ROOT` 或已存在的仓库，否则自动创建 `/content/taskA` 作为实验目录。
不要在 CPU 环境执行加载 cell。
仓库顶层的依赖面向 Python 3.14，本实验只需以下 notebook 专用依赖，沿用 Colab 自带 Python。
下方依赖准备 cell 会在真实推理模式下安装缺失包；也可提前手动安装：

```python
%pip install vllm datasets nbformat
```

可选：如果需要完整仓库，可先克隆再运行配置 cell：

```python
!git clone --branch taskA https://github.com/wangziwei2018/CISC7404-Project.git /content/CISC7404-Project
%cd /content/CISC7404-Project
```

上传本 notebook 也支持相同的独立运行方式。需要自定义保存位置时，在配置 cell **之前**执行：

```python
import os
os.environ["TASK_A_ROOT"] = "/content/taskA"
```

指定目录不存在时会自动创建。`/content` 是临时存储，运行时删除后文件会丢失，请及时下载结果。
模型使用 `Qwen/Qwen3-4B-Instruct-2507`，需要访问 Hugging Face 下载权重；不改成 base 或 Thinking。
推荐把 `TASK_A_ROOT` 设为 Google Drive 中的项目目录以保存断点；本 notebook 不自动挂载 Drive。
若依赖安装 cell 提示重启，只需重启会话后重新从头运行（不要“断开连接并删除运行时”）。
**仅“将 notebook 保存到 GitHub”不会上传运行时内的 JSONL。** 跑完后下载最后一格导出的
`task_a_results.zip`，解压后把 `Experiments/task_a/` 下的结果文件提交到 `taskA`，再保存 notebook。
真实结果：`Experiments/task_a/calib_v3_task.jsonl`；正式数据：
`Experiments/data_generation/synthesis_register_dataset.jsonl`；中间题目和摘要位于 `work/task_a/`。

In [1]:
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import os
import random
import re
import tempfile
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

def project_root():
    if os.environ.get("TASK_A_ROOT"):
        root = Path(os.environ["TASK_A_ROOT"]).expanduser().resolve()
        root.mkdir(parents=True, exist_ok=True)
        return root
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / "task.md").is_file():
            return parent
    content = Path("/content")
    colab = content / "CISC7404-Project"
    if (colab / "task.md").is_file():
        return colab
    if content.is_dir():
        # 从 GitHub 直接打开 Colab 只载入 notebook，不会克隆仓库。
        # 本实验自包含；使用独立输出目录，无需下载其它仓库文件。
        root = content / "taskA"
        root.mkdir(parents=True, exist_ok=True)
        print(f"未发现仓库，使用 Colab 实验目录：{root}。运行时删除后此目录会丢失，请及时保存结果。")
        return root
    raise RuntimeError("请切到仓库目录，或先运行 os.environ['TASK_A_ROOT'] = '可写实验目录'。")

ROOT = project_root()
WORK_DIR = ROOT / "work" / "task_a"
RESULT_PATH = ROOT / "Experiments" / "task_a" / "calib_v3_task.jsonl"
DATA_PATH = ROOT / "Experiments" / "data_generation" / "synthesis_register_dataset.jsonl"
VERSION = "register_v3.1"
SEED = 7404
D_LIST = (1, 5, 10, 15, 20)
RUN_STAGE = "probe"  # @param ["probe", "slope", "offline"]
if RUN_STAGE not in ("probe", "slope", "offline"):
    raise ValueError("RUN_STAGE 只允许 probe / slope / offline")
RUN_PROBES = RUN_STAGE == "probe"
RUN_SLOPE = RUN_STAGE == "slope"
AUTO_DOWNLOAD = True  # Colab 运行结束时自动下载结果包；本地只显示路径
CURRENT_JOBS = []
PROBE_REPORTS = {}
PROBE_REPORT_PATH = RESULT_PATH.parent / "task_a_probe_summary.json"
SLOPE_REPORT_PATH = RESULT_PATH.parent / "task_a_summary.json"
print(f"本次运行阶段：{RUN_STAGE}；" + ("只验证代码，不产生真实校准结果。" if RUN_STAGE == "offline" else "将执行真实模型推理。"))
GENERATE_FINAL = False
PROBE_CANDIDATES = ["register2_small"]  # 首选先跑；看结果后逐个追加其它配置
SELECTED = "register2_small"
PROBE_OVERRIDE_REASON = ""  # p1 不在建议窗口内时，人工给出继续小样本的理由
REVIEW = None  # 看完斜率结果和原文后，在末尾审核 cell 填写
N_FINAL = 100  # 每个串行深度、每个并行宽度的题数；正式题与校准题完全分离

TIER = "B"  # B：4B / T4 正式校准；A：1.7B / RTX3070 仅链路自测
MODEL_REVISION = None  # 建议填写 HF commit；解析出的 commit 会写入运行指纹
PARAMS = {"temperature": 0.6, "top_p": 0.95, "top_k": 20, "max_tokens": 6144}
# top_k 只记录、不传入 SamplingParams，遵守 AGENTS.md。
BATCH_SIZE = 16

def digest(value):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()

def stable_seed(*parts):
    return int(digest(parts)[:8], 16)

def read_jsonl(path):
    if not Path(path).exists():
        return []
    rows = []
    with Path(path).open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"{path}:{line_no} 有不完整 JSON；备份并修复该行后恢复，不静默丢弃。") from exc
    return rows

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # 完整写入后替换，避免生成到一半留下貌似完整的题集。
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    tmp.replace(path)

def roundtrip_questions(path, questions):
    from datasets import load_dataset
    write_jsonl(path, questions)
    # datasets 回读是实际推理的数据入口，不直接用内存中的原始题目。
    ds = load_dataset("json", data_files=str(path), split="train", cache_dir=str(WORK_DIR / "hf_cache"))
    return ds.to_list()

未发现仓库，使用 Colab 实验目录：/content/taskA。运行时删除后此目录会丢失，请及时保存结果。
本次运行阶段：probe；将执行真实模型推理。


## 依赖准备与 GPU 检查

真实探针/斜率模式下先确认有 GPU，再安装缺失的 vLLM / datasets / nbformat。
**安装成功不一定需要重启**：仅当安装实际改变了已加载包的版本，或内存中的包版本与磁盘不一致时，
才显示 `RESTART_REQUIRED` 并停止。提示中会列出具体版本变化。
此时在 Colab 菜单选择 **代码执行程序 → 重新启动会话**，然后再次“全部运行”；
不要选择“断开连接并删除运行时”，否则已安装依赖和输出文件也会被删除。
重启前反复运行本格不会绕过保护；重启后会自动继续。

`offline` 模式只需 datasets，既不安装 vLLM，也不加载模型。
本格通过只是环境就绪，真实成功标志是末尾 **100/100 条真实探针**及 p1 摘要。

In [2]:
def dependency_versions(names):
    versions = {}
    for name in sorted(names):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def loaded_dependency_state():
    import sys
    from packaging.version import InvalidVersion, Version

    package_map = importlib.metadata.packages_distributions()
    roots = {name.split(".")[0] for name in sys.modules}
    names = {dist for root in roots for dist in package_map.get(root, [])}
    installed = dependency_versions(names)
    stale = {}
    # 识别上一轮安装后未重启的会话；只比较可明确对应单一发行包的版本。
    for root in sorted(roots):
        distributions = package_map.get(root, [])
        if len(distributions) != 1:
            continue
        name = distributions[0]
        module = sys.modules.get(root)
        memory = vars(module).get("__version__") if module is not None else None
        disk = installed[name]
        if not isinstance(memory, str) or disk is None:
            continue
        try:
            # PyTorch 的 +cuXXX 后缀可能只出现在模块版本中，不因此误判需重启。
            different = Version(memory).public != Version(disk).public
        except InvalidVersion:
            continue
        if different:
            stale[name] = {"loaded": memory, "installed": disk}
    return installed, stale


def dependency_kernel_token():
    import sys
    import uuid

    # sys 在同一内核内稳定，重跑配置 cell 不会重置；重启后产生新 token。
    if not hasattr(sys, "_task_a_dependency_kernel_token"):
        sys._task_a_dependency_kernel_token = uuid.uuid4().hex
    return sys._task_a_dependency_kernel_token


def prepare_dependencies(required, work_dir=WORK_DIR):
    import subprocess
    import sys

    marker = Path(work_dir) / "dependency_restart_required.json"
    kernel_token = dependency_kernel_token()
    restart_message = (
        "RESTART_REQUIRED：请在 Colab 选择‘代码执行程序 → 重新启动会话’，然后从头运行。"
        "不要删除运行时，也不要只重复执行当前 cell。"
    )
    if marker.is_file():
        previous = json.loads(marker.read_text(encoding="utf-8"))
        if previous["kernel_token"] == kernel_token:
            print("本会话尚未重启：", json.dumps(previous["changes"], ensure_ascii=False))
            raise RuntimeError(restart_message)
        marker.unlink()
        print("已检测到新会话，继续检查依赖。")

    def require_restart(changes):
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps({"kernel_token": kernel_token, "changes": changes},
                                     ensure_ascii=False, indent=2), encoding="utf-8")
        print("需重新加载的包：", json.dumps(changes, ensure_ascii=False, indent=2))
        raise RuntimeError(restart_message)

    before, stale = loaded_dependency_state()
    if stale:
        require_restart(stale)
    missing = [name for name in required if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
        importlib.invalidate_caches()
        after = dependency_versions(before)
        changed = {name: {"before": version, "after": after[name]}
                   for name, version in before.items() if after[name] != version}
        print("依赖安装成功。")
        if changed:
            require_restart(changed)
        print("已加载包的版本未改变，无需重启。")
    print("依赖版本：", json.dumps(dependency_versions(required), ensure_ascii=False))


if RUN_PROBES or RUN_SLOPE:
    import shutil
    import subprocess

    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("未检测到 GPU 驱动。请在 Colab 选择 T4 GPU；仅检查代码时显式选 RUN_STAGE='offline'。")
    gpu_check = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    )
    if not gpu_check.stdout.strip():
        raise RuntimeError("未检测到 GPU。请选择 Colab T4 GPU 后重试。")
    print(gpu_check.stdout.strip())
    prepare_dependencies(("vllm", "datasets", "nbformat"))
    print("GPU 与依赖检查完成；继续运行模型加载和真实探针。")
else:
    print("offline 模式：跳过 GPU 依赖安装。")

Tesla T4, 15360 MiB
依赖版本： {"datasets": "4.8.5", "nbformat": "5.11.1", "vllm": "0.33.0"}
GPU 与依赖检查完成；继续运行模型加载和真实探针。


## 1. 明确定义任务与工作量

`record.md` 的示例混合了单更新和条件更新，与“每步同构”的文字不一致。本版固定每一步：
读取更新前的两个指定寄存器；比较大小；按分支对一个寄存器减 x、另一个加 y，或反向更新。
**两次更新同时生效，等号走 otherwise；x、y 独立且不相等**，避免 `A+B` 守恒导致答案可抄近路。
所有状态在 `[0, base−1]`，不做 mod、截断、min/max 或结果聚合。

默认两位、双寄存器，答案为 `A*100+B`。三寄存器备选追踪额外状态，编码为
`A*10000+B*100+C`；改变位数时 `base=10**digits`，不会使用有歧义的固定乘数。
比较的寄存器对可以不同，但每步都是同一算子。用更多状态/位数调难度，不混入更难算子。
生成器用全值域初态（排除无法合法更新的全零/全上界）和有界拒绝采样，边界会影响操作分布；
必须在校准中检查，不能从 p1 推断成功。

`W` 沿用任务提案的**复合寄存器更新次数**；一次含比较和两次加减，另记录
`W_atomic=3W`。最终答案的无歧义编码是读出协议，不计作更新，串行/并行一致。

并行版采用 `w` 条独立链，长度 `[1,2,...,2]`：`W=2w−1`，真实深度为 `1`（w=1）或 `2`。
与串行 `d=2w−1` 的更新工作量严格相同；**没有为了套公式再添加聚合步骤**。
串行覆盖 d=1..20，并行匹配其中奇数 d=1..19（w=1..10），w=1 是退化对照。
并行答案依次读出所有链的编码，避免漏掉某条链的工作。多答案会增加读出成本，
需同时看 `per_branch_ok`，并在人工审核中评估这一混杂，不能声称只改变了深度。

In [3]:
@dataclass(frozen=True)
class TaskConfig:
    name: str
    digits: int = 2
    registers: int = 2
    delta_max: int = 9

    def __post_init__(self):
        if self.digits not in (1, 2, 3) or self.registers not in (2, 3):
            raise ValueError("只支持 1..3 位、2/3 个寄存器")
        if not 2 <= self.delta_max < 10 ** self.digits:
            raise ValueError("delta_max 必须至少为 2 且小于编码基数")

    @property
    def base(self):
        return 10 ** self.digits

CANDIDATES = {
    c.name: c for c in (
        TaskConfig("register2_small"),
        TaskConfig("register2_wide", delta_max=19),
        TaskConfig("register3_small", registers=3),
        TaskConfig("register2_easy", digits=1, delta_max=3),
    )
}

def encode_state(state, base):
    if any(not 0 <= value < base for value in state):
        raise ValueError("状态越界，不能无歧义编码")
    result = 0
    for value in state:
        result = result * base + value
    return result

def apply_op(state, op):
    out = list(state)
    left, right, x, y = (op[k] for k in ("left", "right", "x", "y"))
    if state[left] > state[right]:
        out[left] -= x
        out[right] += y
    else:
        out[left] += x
        out[right] -= y
    return out

def make_chain(rng, steps, config):
    # 全零 / 全上界没有合法的一加一减；仅排除这两种不可行初态。
    while True:
        state = [rng.randrange(config.base) for _ in range(config.registers)]
        if len(set(state)) > 1 or state[0] not in (0, config.base - 1):
            break
    initial, ops, trace = list(state), [], []
    for _ in range(steps):
        for attempt in range(10000):
            left, right = rng.sample(range(config.registers), 2)
            x, y = rng.sample(range(1, config.delta_max + 1), 2)
            op = {"left": left, "right": right, "x": x, "y": y}
            updated = apply_op(state, op)
            if all(0 <= v < config.base for v in updated):
                break
        else:
            raise RuntimeError("有界生成失败，请检查配置；禁止静默截断或取模")
        ops.append(op)
        trace.append(list(updated))
        state = updated
    return {"initial": initial, "ops": ops, "trace": trace,
            "final": state, "ans": encode_state(state, config.base)}

def op_text(op):
    left, right = ("ABC"[op[k]] for k in ("left", "right"))
    x, y = op["x"], op["y"]
    return (f"If {left} > {right}, subtract {x} from {left} and add {y} to {right}; "
            f"otherwise add {x} to {left} and subtract {y} from {right}.")

def render_question(chains, config):
    names = "ABC"[:config.registers]
    formula = " + ".join(
        name if power == 0 else f"{name}*{config.base ** power}"
        for name, power in zip(names, reversed(range(config.registers)))
    )
    lines = [
        "Process the operations in the listed order within each independent chain.",
        "For each operation, compare the two values BEFORE either update; apply both updates simultaneously.",
        "Equality uses the otherwise branch. Registers not named in an operation stay unchanged.",
        "Use ordinary integer arithmetic, with no wrapping or clipping.",
    ]
    for index, chain in enumerate(chains, 1):
        lines.append(f"Chain {index}: " + ", ".join(f"{n}={v}" for n, v in zip(names, chain["initial"])))
        lines.extend(f"{i}. {op_text(op)}" for i, op in enumerate(chain["ops"], 1))
    lines.append(f"Encode the final registers of each chain as {formula}.")
    lines.append("What is the final encoded integer?" if len(chains) == 1 else
                 "What are the final encoded integers, in chain order? Return all of them, separated by commas.")
    return "\n".join(lines)

def lookup_fields(chains, base):
    # 查表字段只来自题干：不暴露最终状态，不暗中添加 k 轴约束。
    return {
        "START": [encode_state(c["initial"], base) for c in chains],
        "OPS": [len(c["ops"]) for c in chains],
        "FIRST": [op_text(c["ops"][0]) for c in chains],
        "LAST": [op_text(c["ops"][-1]) for c in chains],
        "MAXDELTA": [max(max(op["x"], op["y"]) for op in c["ops"]) for c in chains],
        # 不沿用 min-chain 的 NEG：新任务没有负数初态/字面负增量，定义将空转。
    }

def make_question(config, d, i, split, kind="serial", w=1):
    if not 1 <= d <= 20 or i < 0 or not split:
        raise ValueError("无效的深度、题号或 split")
    if kind == "serial":
        if w != 1:
            raise ValueError("串行宽度必须为 1")
        lengths = [d]
    elif kind == "parallel":
        if not 1 <= w <= 10 or d != (1 if w == 1 else 2):
            raise ValueError("并行深度必须等于最长链长度，w∈1..10")
        lengths = [1] + [2] * (w - 1)
    else:
        raise ValueError("未知任务布局")
    key = {"version": VERSION, "seed": SEED, "config": asdict(config), "split": split,
           "kind": kind, "d": d, "w": w, "i": i}
    rng = random.Random(stable_seed(key))
    chains = [make_chain(rng, steps, config) for steps in lengths]
    workload = sum(lengths)
    answers = [chain["ans"] for chain in chains]
    return {**key, "qid": digest(key), "W": workload, "W_atomic": 3 * workload,
            "matched_serial_d": workload, "chain_lengths": lengths,
            "ans": ",".join(map(str, answers)), "answers": answers,
            "q": render_question(chains, config), "chains": chains,
            "lookup": lookup_fields(chains, config.base)}

## 2. Prompt、答案判分与作业指纹

DA/CoT 只由 system prompt 控制；共同使用 `####ANSWER=` 标记，未加入 k 轴约束。
数值正确性与格式分开：允许最终裸整数/整数列表，空格或 `.0` 不改变数值；
保留多个答案标记、DA 多写推理等诊断，**不据此筛除答案正确的样本**。
没有可辨识最终答案时计错并记录 `parse_ok=False`，不在中间步骤里搜索碰巧等于 gold 的数。

In [4]:
PREAMBLE = {
    "DA": "Solve silently. Do not show calculations, intermediate values, explanations or comments. Output only the final answer line.\n",
    "CoT": "Solve step by step. Explicitly write the register values after every operation, one operation per line, identifying the chain. Then give the final answer.\n",
}
BASE_RULE = "The final line must start with ####ANSWER= followed by the requested encoded integer(s), with nothing after them."

def build_messages(question, mode):
    if mode not in PREAMBLE:
        raise ValueError("只允许 DA / CoT")
    return [{"role": "system", "content": PREAMBLE[mode] + BASE_RULE},
            {"role": "user", "content": question["q"]}]

def parse_answer(text, width):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    marked = re.findall(r"(?m)^\s*####ANSWER\s*=\s*([^\n]*)", text)
    payload = marked[-1].strip() if marked else (lines[-1] if lines else "")
    number = r"[+-]?\d+(?:\.0+)?"
    if not re.fullmatch(number + rf"(?:\s*,\s*{number})*", payload):
        return None
    values = [int(part.strip().split(".")[0]) for part in payload.split(",")]
    return values if len(values) == width else None

def score_answer(text, question, mode):
    values = parse_answer(text, question["w"])
    expected = question["answers"]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    marked = re.findall(r"(?m)^\s*####ANSWER\s*=", text)
    return {
        "answer_ok": values == expected,
        "per_branch_ok": [values is not None and a == b for a, b in zip(values or [None] * len(expected), expected)],
        "parse_ok": values is not None, "predicted": values,
        "base_ok": bool(lines) and lines[-1].startswith("####ANSWER=") and len(marked) == 1 and values is not None,
        "multiple_answers": len(marked) > 1,
        "da_extra_text": mode == "DA" and len(lines) > 1,
    }

def make_jobs(questions, stage, modes, run_meta):
    if stage not in ("probe", "slope"):
        raise ValueError("只支持探针和斜率阶段")
    jobs = []
    for question in questions:
        for mode in modes:
            messages = build_messages(question, mode)
            seed = stable_seed(VERSION, SEED, question["qid"], mode)
            spec = {"stage": stage, "question": question, "mode": mode, "messages": messages,
                    "sampling": PARAMS, "sampling_seed": seed, "run_meta": run_meta}
            jobs.append({**spec, "id": digest(spec)})
    if len({j["id"] for j in jobs}) != len(jobs):
        raise ValueError("作业中有重复题目")
    return jobs

def index_results(path):
    indexed = {}
    for row in read_jsonl(path):
        if row["id"] in indexed:
            raise ValueError(f"结果文件存在重复 id：{row['id']}")
        indexed[row["id"]] = row
    return indexed

def load_complete(jobs, path=RESULT_PATH):
    indexed = index_results(path)
    missing = [job["id"] for job in jobs if job["id"] not in indexed]
    if missing:
        raise RuntimeError(f"当前配置缺少 {len(missing)}/{len(jobs)} 条真实结果，请先跑完对应阶段。")
    rows = [indexed[job["id"]] for job in jobs]
    for job, row in zip(jobs, rows):
        if any(row.get(key) != value for key, value in job.items()):
            raise ValueError("结果与作业指纹不一致")
        if row.get("backend") != "vllm" or job["run_meta"].get("tier") != "B":
            raise ValueError("正式筛选只接受档位 B 的真实 vLLM 结果，离线测试/档位 A 不可定题")
        if any(row.get(key) != value for key, value in score_answer(row["text"], job["question"], job["mode"]).items()):
            raise ValueError("判分记录与原文不一致")
    return rows

## 3. 统计与阶段门槛（不筛掉截断或格式错误）

`task.md` 同时给出 DA(1)=0.80–0.95 与探针窗口 0.955–0.986，两者不相交。
本版分别展示，**不把两者合成不可能通过的硬门槛**。p1 只用作筛选启发式：
DA 单步正确率包含编码/读出错误，不能当作 CoT 的独立同分布逐步成功率。
100 题下窗口只对应 96/97/98 题正确，附 Wilson 95% CI，`p1**20` 仅标为预测。

斜率阶段使用与探针不同的题，每个深度 DA/CoT 配对使用同一道题。汇报
CoT(20)、DA(20)、gap(20)−gap(1)、各深度准确率、解析/截断率和原文。
Cochran–Armitage 使用实际深度分值，对 **CoT 准确率随深度下降** 作双侧检验，
要求 z<0 且 p<0.01；它不是 gap 趋势检验，更不能单凭显著性证明严格单调。
额外报告 gap 的相邻变化及配对 bootstrap 端点差区间，小样本结果须人工判断。

In [5]:
def wilson(successes, n, z=1.959963984540054):
    if n <= 0 or not 0 <= successes <= n:
        raise ValueError("无效二项计数")
    p = successes / n
    denom = 1 + z*z / n
    centre = (p + z*z / (2*n)) / denom
    radius = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return [max(0.0, centre-radius), min(1.0, centre+radius)]

def ca_trend(successes, totals, scores):
    if not (len(successes) == len(totals) == len(scores)) or not totals:
        raise ValueError("趋势数组长度不一致")
    if any(n <= 0 or not 0 <= x <= n for x, n in zip(successes, totals)):
        raise ValueError("无效趋势计数")
    n = sum(totals)
    p = sum(successes) / n
    mean = sum(t*s for t, s in zip(totals, scores)) / n
    variance = p*(1-p)*sum(t*(s-mean)**2 for t, s in zip(totals, scores))
    if variance == 0:
        return {"z": 0.0, "p_two_sided": 1.0, "testable": False}
    z = sum((s-mean)*(x-t*p) for x, t, s in zip(successes, totals, scores)) / math.sqrt(variance)
    return {"z": z, "p_two_sided": math.erfc(abs(z)/math.sqrt(2)), "testable": True}

def probe_summary(rows):
    if len(rows) != 100 or any(r["stage"] != "probe" or r["mode"] != "DA" or r["question"]["d"] != 1 for r in rows):
        raise ValueError("探针必须为 d=1 × 100 × DA")
    hits = sum(r["answer_ok"] for r in rows)
    p1 = hits / 100
    action = ("增加状态或位数" if p1 >= 0.99 else "进入小样本斜率验证" if 0.955 <= p1 <= 0.986
              else "弃用当前配置" if p1 < 0.85 else "偏低，微调配置后重新探针")
    return {"n": 100, "correct": hits, "p1": p1, "wilson95": wilson(hits, 100),
            "predicted_cot20_not_measured": p1**20,
            "in_probe_window": 0.955 <= p1 <= 0.986,
            "in_da1_descriptive_target": 0.80 <= p1 <= 0.95, "action": action,
            "parse_fail_rate": sum(not r["parse_ok"] for r in rows)/100,
            "trunc_rate": sum(r["trunc"] for r in rows)/100}

def slope_summary(rows):
    groups = defaultdict(list)
    for row in rows:
        if row["stage"] != "slope":
            raise ValueError("斜率摘要不能混入探针")
        groups[(row["question"]["d"], row["mode"])].append(row)
    expected_groups = {(d, m) for d in D_LIST for m in ("DA", "CoT")}
    if set(groups) != expected_groups or any(len(group) != 20 for group in groups.values()):
        raise ValueError("斜率阶段须完整覆盖 5 深度 × 20 题 × 2 模式")
    table, paired = [], {}
    for d in D_LIST:
        pair = {m: {r["question"]["qid"]: r for r in groups[d, m]} for m in ("DA", "CoT")}
        if len(pair["DA"]) != 20 or set(pair["DA"]) != set(pair["CoT"]):
            raise ValueError("DA / CoT 必须配对且无重复")
        paired[d] = [int(pair["CoT"][qid]["answer_ok"]) - int(pair["DA"][qid]["answer_ok"]) for qid in sorted(pair["DA"])]
        item = {"d": d}
        for mode in ("DA", "CoT"):
            group = groups[d, mode]
            hits = sum(r["answer_ok"] for r in group)
            item[mode] = {"n": 20, "correct": hits, "accuracy": hits/20, "wilson95": wilson(hits, 20),
                          "trunc_rate": sum(r["trunc"] for r in group)/20,
                          "parse_fail_rate": sum(not r["parse_ok"] for r in group)/20,
                          "mean_tokens": sum(r["n_tokens"] for r in group)/20}
        item["gap"] = sum(paired[d])/20
        table.append(item)
    trends = {m: ca_trend([r[m]["correct"] for r in table], [20]*5, D_LIST) for m in ("DA", "CoT")}
    rng = random.Random(SEED)
    boot = sorted(sum(rng.choices(paired[20], k=20))/20 - sum(rng.choices(paired[1], k=20))/20 for _ in range(5000))
    delta = table[-1]["gap"] - table[0]["gap"]
    checks = {"cot20_hard_target": 0.40 <= table[-1]["CoT"]["accuracy"] <= 0.75,
              "da20_target": table[-1]["DA"]["accuracy"] <= 0.10,
              "gap_growth_target": delta >= 0.30 - 1e-12,
              "cot_decreasing_ca_p_lt_001": trends["CoT"]["z"] < 0 and trends["CoT"]["p_two_sided"] < 0.01}
    evidence = [{"id": r["id"], "text": r["text"], "answer_ok": r["answer_ok"], "trunc": r["trunc"]} for r in sorted(rows, key=lambda r: r["id"])]
    return {"table": table, "ca_accuracy_trends": trends, "gap20_minus_gap1": delta,
            "gap_delta_paired_bootstrap95": [boot[124], boot[4874]],
            "gap_nondecreasing_observed": all(b["gap"] >= a["gap"] - 1e-12 for a, b in zip(table, table[1:])),
            "checks": checks, "evidence_digest": digest(evidence), "manual_review_required": True}

## 4. 离线自测（CPU 可运行，不写模型结果）

独立参考求解器逐步复算所有配置/深度/并行宽度；检查编码、边界、工作量和查表字段。
同时测试答案解析、统计退化情况、阶段完整性、人工审核门槛和断点续跑。
自测中的模拟模型只写临时目录，不能被正式定题流程接受。

In [6]:
def verify_question(question):
    config = TaskConfig(**question["config"])
    actual_lengths = []
    for chain in question["chains"]:
        state = list(chain["initial"])
        assert all(0 <= v < config.base for v in state)
        for i, op in enumerate(chain["ops"]):
            # 独立参考实现：不调用 apply_op/encode_state，防止自测只重复被测实现。
            a, b = op["left"], op["right"]
            old_a, old_b = state[a], state[b]
            sign = 1 if old_a <= old_b else -1
            state[a], state[b] = old_a + sign*op["x"], old_b - sign*op["y"]
            assert op["x"] != op["y"] and a != b
            assert all(0 <= v < config.base for v in state)
            assert state == chain["trace"][i]
        assert state == chain["final"]
        assert sum(v*config.base**i for i, v in enumerate(reversed(state))) == chain["ans"]
        actual_lengths.append(len(chain["ops"]))
    assert question["d"] == max(actual_lengths)
    assert question["W"] == sum(actual_lengths)
    assert question["W_atomic"] == 3 * question["W"]
    assert question["w"] == len(actual_lengths)
    assert question["answers"] == [c["ans"] for c in question["chains"]]
    assert question["ans"] == ",".join(map(str, question["answers"]))
    assert question["lookup"]["OPS"] == actual_lengths
    for j, chain in enumerate(question["chains"]):
        assert question["lookup"]["FIRST"][j] == op_text(chain["ops"][0])
        assert question["lookup"]["LAST"][j] == op_text(chain["ops"][-1])
        assert question["lookup"]["START"][j] == sum(v*config.base**i for i, v in enumerate(reversed(chain["initial"])))
        assert question["lookup"]["MAXDELTA"][j] == max(v for op in chain["ops"] for v in (op["x"], op["y"]))
    if question["kind"] == "parallel":
        assert question["W"] == question["matched_serial_d"] == 2*question["w"]-1
    assert question["q"] == render_question(question["chains"], config)

def expect_error(error, fn):
    try:
        fn()
    except error:
        return
    raise AssertionError(f"预期 {error.__name__}")

def offline_core_tests():
    count = 0
    for config in CANDIDATES.values():
        for d in range(1, 21):
            for i in range(10):
                verify_question(make_question(config, d, i, "selftest"))
                count += 1
        for w in range(1, 11):
            for i in range(10):
                verify_question(make_question(config, 1 if w == 1 else 2, i, "selftest", "parallel", w))
                count += 1
    config = CANDIDATES["register2_small"]
    question = make_question(config, 1, 0, "selftest")
    assert question == make_question(config, 1, 0, "selftest")
    assert question["qid"] != make_question(config, 1, 0, "probe")["qid"]
    assert apply_op([5, 5], {"left": 0, "right": 1, "x": 2, "y": 3}) == [7, 2]
    assert apply_op([6, 5], {"left": 0, "right": 1, "x": 2, "y": 3}) == [4, 8]
    for text in (question["ans"], f"####ANSWER= {question['ans']}.0", f"trace\n####ANSWER={question['ans']}"):
        assert score_answer(text, question, "DA")["answer_ok"]
    for text in ("", "A=41 B=18", "####ANSWER=12 or 34", "####ANSWER=1.5", "####ANSWER=1,2"):
        assert parse_answer(text, 1) is None
    assert parse_answer("####ANSWER=1\n####ANSWER=2", 1) == [2]
    assert parse_answer("####ANSWER=1, 2", 2) == [1, 2]
    assert parse_answer("####ANSWER=1", 2) is None
    assert not score_answer("####ANSWER=-1", question, "DA")["answer_ok"]
    assert not score_answer(f"trace\n{question['ans']}", question, "DA")["base_ok"]
    assert ca_trend([20]*5, [20]*5, D_LIST)["p_two_sided"] == 1
    assert math.isclose(ca_trend([0, 10], [10, 10], [1, 2])["z"], math.sqrt(20))
    trend = ca_trend([20, 18, 14, 10, 8], [20]*5, D_LIST)
    assert trend["z"] < 0 and trend["p_two_sided"] < .01
    reverse = ca_trend([0, 2, 6, 10, 12], [20]*5, D_LIST)
    assert math.isclose(trend["z"], -reverse["z"])
    lo, hi = wilson(96, 100)
    assert .90 < lo < .96 < hi < 1
    expect_error(ValueError, lambda: probe_summary([]))
    expect_error(ValueError, lambda: slope_summary([]))
    expect_error(ValueError, lambda: make_question(config, 3, 0, "test", "parallel", 3))
    print(f"离线核心自测通过：{count} 道题逐步复算；无模型推理，无校准结论。")

offline_core_tests()

离线核心自测通过：1200 道题逐步复算；无模型推理，无校准结论。


## 5. 加载 vLLM（仅在开启探针或斜率时执行）

B 使用历史校准已验证的 T4 模板：half / 0.90 / 8192 / 32，不传 thinking kwargs。
A 原样沿用 AGENTS 的 0.80 / 8192 模板，并显式 `enable_thinking=False`。
两种模式共用 6144 输出 token，生成原文不做后处理截断。每题独立采样种子使续跑不依赖批次顺序。
实际依赖版本、GPU、模型 commit、加载参数进入运行指纹，避免混用不同运行环境的结果。

In [7]:
def load_vllm():
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"  # 必须先于 import vllm
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("本机没有 CUDA；保留离线验证，请在 Colab/T4 运行真实探针。")
    from vllm import LLM, SamplingParams
    model_id = {"A": "Qwen/Qwen3-1.7B", "B": "Qwen/Qwen3-4B-Instruct-2507"}[TIER]
    local = ROOT / "models" / model_id.split("/")[-1]
    source = str(local) if local.is_dir() else model_id
    if TIER == "B":
        if torch.cuda.get_device_properties(0).total_memory / 2**30 < 14:
            raise RuntimeError("档位 B 使用 T4/至少 14 GiB 可见显存；8GB 卡请用 A 做链路自测。")
        loading = dict(model=source, dtype="half", gpu_memory_utilization=0.90,
                       max_model_len=8192, max_num_seqs=32)
        chat_kw = {}
    else:
        loading = dict(model=source, gpu_memory_utilization=0.80, max_model_len=8192)
        chat_kw = {"chat_template_kwargs": {"enable_thinking": False}}
    if MODEL_REVISION is not None:
        loading["revision"] = MODEL_REVISION
    model = LLM(**loading)
    hf_config = model.llm_engine.model_config.hf_config
    model_commit = getattr(hf_config, "_commit_hash", None)
    if model_commit is None:
        # 本地权重必须显式标记版本，不能仅凭目录名混合不同权重的断点。
        if MODEL_REVISION is None:
            raise RuntimeError("无法解析模型 commit，请填写 MODEL_REVISION 为权重版本后重启内核。")
        model_commit = MODEL_REVISION
    meta = {"tier": TIER, "model_id": model_id, "model_commit": model_commit,
            "loading": loading, "chat_kwargs": chat_kw,
            "gpu": torch.cuda.get_device_name(0), "top_k_passed": False,
            "packages": {p: importlib.metadata.version(p) for p in ("vllm", "torch", "transformers", "datasets")}}
    return model, SamplingParams, chat_kw, meta

if RUN_PROBES or RUN_SLOPE:
    if "llm" not in globals():
        llm, SamplingParams, CHAT_KW, RUN_META = load_vllm()
    else:
        if RUN_META["tier"] != TIER or RUN_META["loading"].get("revision") != MODEL_REVISION:
            raise RuntimeError("模型配置改变，请重启内核再加载，避免复用旧权重。")
    print(json.dumps(RUN_META, ensure_ascii=False, indent=2))
else:
    print("offline 模式：GPU 加载已跳过。真实推理请选择 RUN_STAGE='probe' 或 'slope' 后从头运行。")

W1108 18:11:30.600000 10852 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1108 18:11:30.602000 10852 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1108 18:11:30.603000 10852 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=10852) INFO 11-08 18:11:52 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 32, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
(MainProcess pid=10852) INFO 11-08 18:11:53 [model.py:744] Resolved architecture: Qwen3ForCausalLM
(MainProcess pid=10852) WARNING 11-08 18:11:53 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=10852) INFO 11-08 18:11:53 [model.py:2170] Using max model len 8192
(MainProcess pid=10852) INFO 11-08 18:11:53 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

(MainProcess pid=10852) INFO 11-08 18:11:54 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

(MainProcess pid=10852) WARNING 11-08 18:12:04 [system_utils.py:154] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=10852) INFO 11-08 18:13:28 [hf.py:736] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


RuntimeError: 无法解析模型 commit，请填写 MODEL_REVISION 为权重版本后重启内核。

In [ ]:
def run_jobs(jobs, model, sampling_class, chat_kw, path=RESULT_PATH, backend="vllm"):
    path = Path(path)
    indexed = index_results(path)
    if len({job["id"] for job in jobs}) != len(jobs):
        raise ValueError("不能运行重复作业")
    for job in jobs:
        if job["id"] in indexed:
            previous = indexed[job["id"]]
            if previous.get("backend") != backend or any(previous.get(k) != v for k, v in job.items()):
                raise ValueError("已有结果与当前作业不一致，拒绝覆盖")
    pending = [job for job in jobs if job["id"] not in indexed]
    if not pending:
        print(f"已完成 {len(jobs)} 条，无需重复推理。")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer = model.get_tokenizer()
    for offset in range(0, len(pending), BATCH_SIZE):
        batch = pending[offset:offset+BATCH_SIZE]
        for job in batch:
            tokens = tokenizer.apply_chat_template(job["messages"], tokenize=True, add_generation_prompt=True,
                                                   **chat_kw.get("chat_template_kwargs", {}))
            if len(tokens) + job["sampling"]["max_tokens"] > 8192:
                raise ValueError("题干与输出预算超过上下文；禁止自动截断，先修改设计并重校准。")
        params = [sampling_class(temperature=j["sampling"]["temperature"], top_p=j["sampling"]["top_p"],
                                 max_tokens=j["sampling"]["max_tokens"], seed=j["sampling_seed"]) for j in batch]
        outputs = model.chat([job["messages"] for job in batch], params, use_tqdm=False, **chat_kw)
        if len(outputs) != len(batch) or any(len(o.outputs) != 1 for o in outputs):
            raise RuntimeError("推理返回数量不匹配，当前批次未写入")
        with path.open("a", encoding="utf-8") as f:
            for job, output in zip(batch, outputs):
                result = output.outputs[0]
                row = {**job, "backend": backend, "text": result.text,
                       "finish_reason": result.finish_reason, "stop_reason": result.stop_reason,
                       "n_tokens": len(result.token_ids), "prompt_tokens": len(output.prompt_token_ids),
                       "trunc": result.finish_reason == "length", "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                       **score_answer(result.text, job["question"], job["mode"])}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
            os.fsync(f.fileno())
        print(f"新增 {min(offset+BATCH_SIZE, len(pending))}/{len(pending)} 条")

def prepare_jobs(config, stage, run_meta):
    if stage == "probe":
        questions = [make_question(config, 1, i, "probe") for i in range(100)]
        modes = ("DA",)
    elif stage == "slope":
        questions = [make_question(config, d, i, "slope") for d in D_LIST for i in range(20)]
        modes = ("DA", "CoT")
    else:
        raise ValueError("未知阶段")
    path = WORK_DIR / f"{stage}_{digest([VERSION, SEED, asdict(config)])[:16]}.jsonl"
    loaded = roundtrip_questions(path, questions)
    assert loaded == questions
    return make_jobs(loaded, stage, modes, run_meta)

## 6. A.1 探针：先首选，再调难度

第一次只跑 `register2_small`。默认调参顺序为增加增量位数、增加寄存器状态；如果偏难可用
`register2_easy`。这些是候选配置，**不是已验证的可用任务**。如果本任务族均不合适，
按 task.md 再研究编号数组指针/槽位覆盖写，不自动挑出最接近目标的失败配置。

In [ ]:
if RUN_PROBES:
    for name in PROBE_CANDIDATES:
        jobs = prepare_jobs(CANDIDATES[name], "probe", RUN_META)
        CURRENT_JOBS = list({j["id"]: j for j in CURRENT_JOBS + jobs}.values())
        run_jobs(jobs, llm, SamplingParams, CHAT_KW)
        if TIER == "B":
            summary = probe_summary(load_complete(jobs))
            PROBE_REPORTS[name] = {"config": asdict(CANDIDATES[name]), "summary": summary}
            report = {"version": VERSION, "stage": "probe", "run_meta": RUN_META,
                      "candidates": PROBE_REPORTS, "formal_generation_approved": False}
            PROBE_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
            PROBE_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
            print(name, json.dumps(summary, ensure_ascii=False, indent=2))
            print(f"真实探针完成：{len(jobs)}/100 条；p1={summary['p1']:.3f}。下一步：{summary['action']}。")
        else:
            print("档位 A 仅链路自测；不能使用这些结果定题。")
else:
    print(f"当前阶段 {RUN_STAGE}：未执行新的探针。")

## 7. A.3 小样本斜率验证

必须先完成所选配置的 100 条 B 档探针。窗口外若仍值得探索，在 `PROBE_OVERRIDE_REASON`
记录理由；这只是允许跑 200 条小样本，不代表批准正式生成。
先保留上轮的 `calib_v3_task.jsonl`，再改 `RUN_STAGE="slope"` 并从头运行。
若更换了 Colab 运行时，请先把结果包中的 JSONL 恢复到 `RESULT_PATH`；仅保存 notebook 不会保存断点。

In [ ]:
if RUN_SLOPE:
    probe_jobs = prepare_jobs(CANDIDATES[SELECTED], "probe", RUN_META)
    selected_probe = probe_summary(load_complete(probe_jobs))
    if not selected_probe["in_probe_window"] and not PROBE_OVERRIDE_REASON.strip():
        raise RuntimeError("p1 不在 0.955–0.986；先微调，或填写继续小样本的人工理由。")
    slope_jobs = prepare_jobs(CANDIDATES[SELECTED], "slope", RUN_META)
    CURRENT_JOBS = list({j["id"]: j for j in CURRENT_JOBS + slope_jobs}.values())
    run_jobs(slope_jobs, llm, SamplingParams, CHAT_KW)
    slope_rows = load_complete(slope_jobs)
    selected_slope = slope_summary(slope_rows)
    report = {"version": VERSION, "candidate": SELECTED, "config": asdict(CANDIDATES[SELECTED]),
              "run_meta": RUN_META, "probe": selected_probe, "slope": selected_slope,
              "probe_override_reason": PROBE_OVERRIDE_REASON}
    SLOPE_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    SLOPE_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print("人工审核：检查摘要以及下方样例；完整原文见 calib_v3_task.jsonl。")
    for d in D_LIST:
        for mode in ("DA", "CoT"):
            group = [r for r in slope_rows if r["question"]["d"] == d and r["mode"] == mode]
            for target in (True, False):
                row = next((r for r in group if r["answer_ok"] == target), None)
                if row:
                    print(f"\nd={d} {mode} answer_ok={target} gold={row['question']['ans']} qid={row['question']['qid']}")
                    print(row["text"])
else:
    print(f"当前阶段 {RUN_STAGE}：未执行新的斜率验证；须完成探针后才进入 slope 阶段。")

## 8. 人工审核与正式生成（默认锁定）

**提醒开发者：task.md A.0 明确要求出结果后人工审查。** 检查 100 条探针、配对斜率曲线、
错误样例、截断/解析失败率、是否存在捷径，以及并行多答案读出成本。
CoT(20)∈[0.40,0.75] 保留为硬门槛；其余目标未满足时必须在 note 解释选择理由。
批准绑定到原始结果摘要指纹，换模型/配置/结果后需重新审核。

审查后在此 cell 下方自行填写（不要照抄空值，`approved` 默认为 False）：

```python
REVIEW = {
    "approved": False,
    "reviewer": "",
    "candidate": SELECTED,
    "evidence_digest": selected_slope["evidence_digest"],
    "note": "",  # 写清错误模式、未达目标及并行对照混杂的审查结论
}
GENERATE_FINAL = True
```

若重启内核后仅生成数据，不需要重新加载 GPU：从 `Experiments/task_a/task_a_summary.json` 读取
`run_meta` 赋给 `RUN_META`，重建对应作业并校验已有原文即可。正式 JSONL 的 `ans` 一律为字符串，
`answers` 为整数列表；`d` 永远表示实际最长依赖链，配平用 `matched_serial_d` / `W`。
生成后用 datasets 回读，再独立复算；汇总必须回填 record.md 与 AGENTS.md 5.5。

In [ ]:
def require_review(review, candidate, summary):
    if not summary["checks"]["cot20_hard_target"]:
        raise ValueError("CoT(d=20) 不在硬目标区间；继续调题，不能正式生成。")
    if not isinstance(review, dict) or review.get("approved") is not True:
        raise ValueError("缺少人工批准；不得先造正式数据集。")
    if review.get("candidate") != candidate or review.get("evidence_digest") != summary["evidence_digest"]:
        raise ValueError("人工审核与当前候选/结果不匹配")
    if not str(review.get("reviewer") or "").strip() or not str(review.get("note") or "").strip():
        raise ValueError("请填写审查人及审查结论")

def formal_questions(config, n):
    if not isinstance(n, int) or n < 1:
        raise ValueError("每格题数必须为正整数")
    rows = [make_question(config, d, i, "formal") for d in range(1, 21) for i in range(n)]
    rows.extend(make_question(config, 1 if w == 1 else 2, i, "formal", "parallel", w)
                for w in range(1, 11) for i in range(n))
    return rows

def generate_final(config, run_meta, review, n=N_FINAL, result_path=RESULT_PATH, output_path=DATA_PATH):
    probe_jobs = prepare_jobs(config, "probe", run_meta)
    slope_jobs = prepare_jobs(config, "slope", run_meta)
    probe = probe_summary(load_complete(probe_jobs, result_path))
    slope = slope_summary(load_complete(slope_jobs, result_path))
    require_review(review, config.name, slope)
    rows = formal_questions(config, n)
    calibration_qids = {j["question"]["qid"] for j in probe_jobs+slope_jobs}
    calibration_prompts = {j["question"]["q"] for j in probe_jobs+slope_jobs}
    if any(r["qid"] in calibration_qids or r["q"] in calibration_prompts for r in rows):
        raise ValueError("正式题与校准题重叠，请调整正式 split 后再生成")
    if len({r["q"] for r in rows}) != len(rows):
        raise ValueError("正式题重复；请调整正式 split 或题量后再生成")
    for row in rows:
        verify_question(row)
    # 防止无意覆盖其它配置的正式产物。相同题目重复生成可安全复用。
    path = Path(output_path)
    if path.exists() and read_jsonl(path) != rows:
        raise FileExistsError("正式文件已存在且内容不同，请先归档旧版或使用新输出路径")
    loaded = roundtrip_questions(path, rows)
    assert loaded == rows
    for row in loaded:
        verify_question(row)
    manifest = {"version": VERSION, "config": asdict(config), "run_meta": run_meta,
                "review": review, "probe": probe, "slope": slope, "n_per_cell": n,
                "serial_rows": 20*n, "parallel_rows": 10*n, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    path.with_suffix(".manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"正式生成并回读验证 {len(loaded)} 条：串行 {20*n}，并行 {10*n}。")
    return manifest

if GENERATE_FINAL:
    if "RUN_META" not in globals():
        raise RuntimeError("请从真实运行摘要恢复 RUN_META，不能凭空构造实验来源。")
    final_manifest = generate_final(CANDIDATES[SELECTED], RUN_META, REVIEW)
else:
    print("正式生成关闭；无实测与人工审核时不创建 synthesis_register_dataset.jsonl。")

In [ ]:
def offline_pipeline_tests():
    from types import SimpleNamespace
    config = CANDIDATES["register2_small"]
    question = make_question(config, 1, 0, "pipeline_test")
    meta = {"tier": "B", "test_only": True}
    jobs = make_jobs([question], "probe", ("DA", "CoT"), meta)
    assert len(jobs) == 2 and jobs[0]["question"] == jobs[1]["question"]
    assert jobs[0]["messages"][1] == {"role": "user", "content": question["q"]}
    assert jobs[0]["id"] != make_jobs([question], "probe", ("DA",), {"tier": "A"})[0]["id"]

    class FakeModel:
        calls = 0
        def get_tokenizer(self):
            return SimpleNamespace(apply_chat_template=lambda *args, **kwargs: [1, 2, 3])
        def chat(self, messages, params, **kwargs):
            self.calls += 1
            assert all("top_k" not in p for p in params)
            return [SimpleNamespace(prompt_token_ids=[1, 2, 3], outputs=[SimpleNamespace(
                text=f"####ANSWER={question['ans']}", finish_reason="stop", stop_reason=None, token_ids=[4, 5])]) for _ in messages]

    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "mock.jsonl"
        model = FakeModel()
        run_jobs(jobs[:1], model, dict, {}, path, backend="offline_mock")
        run_jobs(jobs, model, dict, {}, path, backend="offline_mock")
        run_jobs(jobs, model, dict, {}, path, backend="offline_mock")
        assert model.calls == 2 and len(read_jsonl(path)) == 2
        expect_error(ValueError, lambda: load_complete(jobs, path))
        expect_error(RuntimeError, lambda: load_complete(jobs, Path(tmp)/"missing.jsonl"))
        expect_error(ValueError, lambda: run_jobs(jobs+jobs, model, dict, {}, path, backend="offline_mock"))
        class LongPromptModel(FakeModel):
            def get_tokenizer(self):
                return SimpleNamespace(apply_chat_template=lambda *args, **kwargs: [1]*2049)
        expect_error(ValueError, lambda: run_jobs(jobs, LongPromptModel(), dict, {}, Path(tmp)/"long.jsonl", backend="offline_mock"))
        duplicate_path = Path(tmp)/"duplicate.jsonl"
        write_jsonl(duplicate_path, read_jsonl(path)*2)
        expect_error(ValueError, lambda: index_results(duplicate_path))
        # JSONL → datasets 的嵌套字段、答案类型和工作量回读验证。
        sample = formal_questions(config, 2)
        assert len(sample) == 60
        loaded = roundtrip_questions(Path(tmp)/"questions.jsonl", sample)
        assert loaded == sample
        for row in loaded:
            verify_question(row)
        path.write_text('{"id":', encoding="utf-8")
        expect_error(ValueError, lambda: read_jsonl(path))

    summary = {"checks": {"cot20_hard_target": True}, "evidence_digest": "test-only"}
    review = {"approved": True, "reviewer": "离线测试", "note": "不是真实审核", "candidate": config.name, "evidence_digest": "test-only"}
    expect_error(ValueError, lambda: require_review(None, config.name, summary))
    expect_error(ValueError, lambda: require_review({**review, "evidence_digest": "old"}, config.name, summary))
    expect_error(ValueError, lambda: require_review(review, config.name, {**summary, "checks": {"cot20_hard_target": False}}))
    require_review(review, config.name, summary)
    rows = []
    for depth, da_hits, cot_hits in zip(D_LIST, (18, 10, 4, 1, 0), (20, 18, 16, 13, 10)):
        for i in range(20):
            q = make_question(config, depth, i, "stats_test")
            for mode, hits in (("DA", da_hits), ("CoT", cot_hits)):
                rows.append({"id": f"{depth}-{i}-{mode}", "stage": "slope", "mode": mode, "question": q,
                             "text": "模拟数据，绝不落盘", "answer_ok": i < hits, "trunc": False, "parse_ok": True, "n_tokens": 1})
    result = slope_summary(rows)
    assert all(result["checks"].values()) and math.isclose(result["gap20_minus_gap1"], .4)
    expect_error(ValueError, lambda: slope_summary(rows[:-1]))
    print("离线流程自测通过：断点续跑、去重、回读、配对统计、拒绝模拟结果与审核门槛。")

offline_pipeline_tests()

## 9. 运行后的交付清单

- **不要只上传 notebook**。下载下方结果包，解压后把 JSONL、摘要和导出清单提交到 GitHub。
- 将真实 `calib_v3_task.jsonl` 连同 `Experiments/task_a/task_a_summary.json` 保存；保留失败配置，不能只交入选题。
- **人工审查** p1 / 曲线 / 错误原文及并行定义；没有通过就继续探针，不生成正式文件。
- 通过后保存 `synthesis_register_dataset.jsonl` 与同名 `.manifest.json`；默认 2000 串行 + 1000 并行。
- 将模型 commit、参数、样本数、p1 CI、CoT(20)、DA(20)、gap 差、CA 检验对象和 p 值回填
  `record.md` 与 `AGENTS.md` 5.5；把 task.md 对应复选框标为完成。
- 查表字段已预留 START/OPS/FIRST/LAST/MAXDELTA；NEG 不适用于本任务，交由 Task B 重定义并重标定。
- 不运行完整 k 矩阵；本次代码交付不是校准通过结论。

## 10. 真实结果核对与导出

只保存 notebook 不能交付模型原文。本格核对当前阶段每条作业与真实 JSONL 的一致性，
输出完成数量和下一步，并生成 `task_a_results.zip`；Colab 默认触发下载。
ZIP 中保留仓库相对路径，将其解压后的 `Experiments/task_a/` 内容提交到 `taskA`。
如果中途推理失败，可以单独运行本格保存已完成的部分；未完成状态会明确标记，不能当作完整校准。
`offline` 模式不会生成结果包，模拟自测不会增加真实记录数。

In [ ]:
def export_result_bundle(jobs, stage, root=ROOT, result_path=RESULT_PATH, report_paths=()):
    import zipfile

    if stage not in ("probe", "slope") or not jobs:
        raise ValueError("没有当前阶段的真实推理作业，不生成结果包。")
    if any(job["stage"] != stage for job in jobs):
        raise ValueError("导出作业阶段与 RUN_STAGE 不一致，请重新运行配置 cell。")
    indexed = index_results(result_path)
    completed_jobs = [job for job in jobs if job["id"] in indexed]
    if not completed_jobs:
        raise RuntimeError("当前阶段尚无真实推理结果；离线自测不算探针。")
    # 复算判分、核对提示词/采样/模型版本，拒绝混入模拟结果。
    load_complete(completed_jobs, result_path)
    if any(row.get("backend") != "vllm" or row["run_meta"].get("tier") != "B" for row in indexed.values()):
        raise ValueError("结果文件混有非 B 档真实推理记录，请分开归档后再导出。")
    if len({job["id"] for job in jobs}) != len(jobs):
        raise ValueError("当前导出作业有重复 id。")
    root = Path(root).resolve()
    result_path = Path(result_path).resolve()
    result_path.relative_to(root)
    archive_path = result_path.parent / "task_a_results.zip"
    manifest_path = result_path.parent / "task_a_export_manifest.json"
    manifest = {"version": VERSION, "stage": stage, "expected_current_jobs": len(jobs),
                "completed_current_jobs": len(completed_jobs), "stage_complete": len(completed_jobs) == len(jobs),
                "total_saved_real_results": len(indexed), "current_job_ids": [job["id"] for job in jobs],
                "result_sha256": hashlib.sha256(result_path.read_bytes()).hexdigest(),
                "formal_generation_approved": False}
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    payload = [result_path, manifest_path]
    payload.extend(Path(p).resolve() for p in report_paths if Path(p).is_file())
    for item in payload:
        item.relative_to(root)
    tmp = archive_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for item in payload:
            archive.write(item, arcname=item.relative_to(root).as_posix())
    tmp.replace(archive_path)
    print(f"真实 {stage} 结果：{len(completed_jobs)}/{len(jobs)} 条；完整阶段：{manifest['stage_complete']}。")
    print(f"结果包：{archive_path}。请解压并将 Experiments/task_a/ 下的结果文件提交到 GitHub。")
    return archive_path, manifest

if RUN_STAGE == "offline":
    print("本次没有新的模型推理；离线自测本身不能判断 Task A 是否达标。")
elif TIER != "B":
    print("档位 A 仅链路自测，不导出为 B 档正式校准结果。")
elif CURRENT_JOBS:
    bundle_path, export_manifest = export_result_bundle(
        CURRENT_JOBS, RUN_STAGE, report_paths=(PROBE_REPORT_PATH, SLOPE_REPORT_PATH),
    )
    if AUTO_DOWNLOAD and Path("/content").is_dir():
        try:
            from google.colab import files
        except ImportError:
            print("非 Colab 环境，请从上述路径取走 ZIP。")
        else:
            files.download(str(bundle_path))
else:
    print("当前阶段尚未提交任何推理作业；请先运行对应的探针/斜率 cell。")